In [1]:
import pandas as pd
df = pd.read_parquet(r"C:\QP2\data\meta\sp500_universe.parquet")
print(df.columns.tolist())
print(df.head())

['ticker_sp', 'ticker_yahoo', 'cik', 'security_name', 'GICS Sector', 'GICS Sub-Industry']
  ticker_sp ticker_yahoo         cik        security_name  \
0       MMM          MMM  0000066740                   3M   
1       AOS          AOS  0000091142          A. O. Smith   
2       ABT          ABT  0000001800  Abbott Laboratories   
3      ABBV         ABBV  0001551152               AbbVie   
4       ACN          ACN  0001467373            Accenture   

              GICS Sector               GICS Sub-Industry  
0             Industrials        Industrial Conglomerates  
1             Industrials               Building Products  
2             Health Care           Health Care Equipment  
3             Health Care                   Biotechnology  
4  Information Technology  IT Consulting & Other Services  


In [3]:
import pandas as pd
import requests
from io import StringIO

url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
resp = requests.get(url, headers=headers)
tables = pd.read_html(StringIO(resp.text))
changes = tables[1]
changes.to_csv(r"C:\QP2\data\meta\sp500_changes.csv", index=False)
print(changes.columns.tolist())
print(changes.shape)
print(changes.head(10))

[('Effective Date', 'Effective Date'), ('Added', 'Ticker'), ('Added', 'Security'), ('Removed', 'Ticker'), ('Removed', 'Security'), ('Reason', 'Reason')]
(389, 6)
      Effective Date  Added                              Removed  \
      Effective Date Ticker                     Security  Ticker   
0   February 9, 2026   CIEN                        Ciena     DAY   
1  December 22, 2025    CRH                          CRH     LKQ   
2  December 22, 2025   CVNA                      Carvana    SOLS   
3  December 22, 2025    FIX          Comfort Systems USA     MHK   
4  December 11, 2025   ARES              Ares Management       K   
5  November 28, 2025   SNDK                      Sandisk     IPG   
6   November 4, 2025    NaN                          NaN     EMN   
7   November 3, 2025      Q            Qnity Electronics     NaN   
8   October 31, 2025    NaN                          NaN     KMX   
9   October 30, 2025   SOLS  Solstice Advanced Materials     NaN   

                     

In [4]:
# 컬럼 정리
changes.columns = ['date', 'added_ticker', 'added_name', 'removed_ticker', 'removed_name', 'reason']
changes['date'] = pd.to_datetime(changes['date'])
changes = changes.sort_values('date').reset_index(drop=True)

print(f"기간: {changes['date'].min()} ~ {changes['date'].max()}")
print(f"총 {len(changes)}건")
print(f"\n최초 10건:")
print(changes.head(10).to_string())
print(f"\n최근 10건:")
print(changes.tail(10).to_string())

# 저장 (정리된 버전)
changes.to_parquet(r"C:\QP2\data\meta\sp500_changes.parquet", index=False)
print("\n✅ 저장 완료: sp500_changes.parquet")

기간: 1976-07-01 00:00:00 ~ 2026-02-09 00:00:00
총 389건

최초 10건:
        date added_ticker                     added_name removed_ticker              removed_name                                                                                                                                       reason
0 1976-07-01          DIS        The Walt Disney Company            AYE          Allegheny Energy  Major restructuring of S&P 500 to have fewer industrials and utilities, and more financial companies to add "new strength and breadth"[273]
1 1976-07-01          BUD                 Anheuser Busch            HNG       Houston Natural Gas  Major restructuring of S&P 500 to have fewer industrials and utilities, and more financial companies to add "new strength and breadth"[273]
2 1994-09-30          NCC                  National City            MCK                  McKesson                                                                                         McKesson sold PCS Health Services t

In [5]:
# =============================================================================
# 🔨 07_Test2_Survive — 셀 0: Point-in-Time S&P500 유니버스 구축
# =============================================================================
#
# [목적]
# 현재 S&P500 구성종목을 과거에 소급 적용하는 생존편향을 제거.
# 위키피디아 S&P500 변경 이력(sp500_changes.parquet)을 사용하여
# 각 월별로 "그 시점의 실제 S&P500 구성종목"을 역산.
#
# [로직]
# 1. 기준점: 2026-02 현재 S&P500 = sp500_universe.parquet (503종목)
# 2. sp500_changes를 최신→과거 역순으로 탐색:
#    - added_ticker가 있으면 → 해당 날짜 이전에는 유니버스에서 제거
#      (편입 전이니까 S&P500이 아니었음)
#    - removed_ticker가 있으면 → 해당 날짜 이전에는 유니버스에 추가
#      (퇴출 전이니까 S&P500이었음)
# 3. 월말 기준으로 스냅샷 생성
#
# [산출물]
# data/interim/sp500_pit_universe.parquet
#   컬럼: date, ticker_yahoo
#   각 행 = "이 날짜에 이 종목이 S&P500 구성종목이었다"
#
# [주의]
# - 위키피디아 데이터가 100% 완벽하지 않을 수 있음 (누락, 오타)
# - ticker 변경(리네이밍)은 반영 안 됨 — 추후 수동 보정 필요 시 대응
# - 1976년부터 이력 있으나 백테스트 시작은 2013-06 → 2013년 이전은 참고용
# =============================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm

QP2_ROOT = Path(r"C:\QP2")
DATA_DIR = QP2_ROOT / "data"
META_DIR = DATA_DIR / "meta"
INTERIM  = DATA_DIR / "interim"

# ─── 1. 데이터 로드 ───────────────────────────────────────────────────────────

# 현재 S&P500 구성종목
universe = pd.read_parquet(META_DIR / "sp500_universe.parquet")
current_tickers = set(universe["ticker_yahoo"].dropna().unique())
print(f"현재 S&P500: {len(current_tickers)}종목")

# 변경 이력
changes = pd.read_parquet(META_DIR / "sp500_changes.parquet")
changes["date"] = pd.to_datetime(changes["date"])
changes = changes.sort_values("date", ascending=False).reset_index(drop=True)  # 최신→과거
print(f"변경 이력: {len(changes)}건 ({changes['date'].min().date()} ~ {changes['date'].max().date()})")

# ─── 2. 변경 이력에서 ticker 매핑 확인 ────────────────────────────────────────

# Yahoo ticker와 S&P ticker가 다를 수 있음 (BRK.B vs BRK-B 등)
# sp500_universe에 ticker_sp, ticker_yahoo 둘 다 있으니 매핑 생성
sp_to_yahoo = universe.set_index("ticker_sp")["ticker_yahoo"].to_dict()

def to_yahoo_ticker(sp_ticker):
    """S&P ticker → Yahoo ticker 변환. 매핑 없으면 . → - 치환."""
    if pd.isna(sp_ticker) or sp_ticker == "":
        return None
    sp_ticker = str(sp_ticker).strip()
    if sp_ticker in sp_to_yahoo:
        return sp_to_yahoo[sp_ticker]
    # . → - 치환 (BRK.B → BRK-B)
    yahoo_guess = sp_ticker.replace(".", "-")
    return yahoo_guess

# ─── 3. 역순 탐색으로 월별 유니버스 구축 ──────────────────────────────────────

# 백테스트 기간 정의
bt_start = pd.Timestamp("2010-01-01")  # 여유분 (팩터 계산에 lookback 필요)
bt_end = pd.Timestamp("2026-02-28")

# 월말 날짜 생성
monthly_dates = pd.date_range(bt_start, bt_end, freq="ME")

# 시작점: 현재 유니버스 복사
live_set = current_tickers.copy()

# 변경 이력을 날짜 기준으로 그룹핑 (역순)
changes_sorted = changes.sort_values("date", ascending=False)

# 날짜별 변경 사항 딕셔너리 (날짜 → [(added, removed), ...])
change_events = {}
for _, row in changes_sorted.iterrows():
    dt = row["date"]
    added = to_yahoo_ticker(row["added_ticker"])
    removed = to_yahoo_ticker(row["removed_ticker"])
    if dt not in change_events:
        change_events[dt] = []
    change_events[dt].append({"added": added, "removed": removed})

# 역순으로 월별 스냅샷 생성
# 현재 → 과거로 가면서, 각 변경 이벤트를 "되돌림"
snapshots = {}
event_dates = sorted(change_events.keys(), reverse=True)
event_idx = 0

for dt in tqdm(sorted(monthly_dates, reverse=True), desc="PIT Universe"):
    # 이 월말보다 이후이고 아직 처리 안 한 변경 이벤트 적용
    while event_idx < len(event_dates) and event_dates[event_idx] > dt:
        for event in change_events[event_dates[event_idx]]:
            # "되돌림": 편입(added) → 이전에는 없었으니 제거
            if event["added"] and event["added"] in live_set:
                live_set.remove(event["added"])
            # "되돌림": 퇴출(removed) → 이전에는 있었으니 추가
            if event["removed"]:
                live_set.add(event["removed"])
        event_idx += 1

    snapshots[dt] = live_set.copy()

# ─── 4. DataFrame 변환 ───────────────────────────────────────────────────────

records = []
for dt in sorted(snapshots.keys()):
    for ticker in sorted(snapshots[dt]):
        records.append({"date": dt, "ticker_yahoo": ticker})

pit_universe = pd.DataFrame(records)
pit_universe["date"] = pd.to_datetime(pit_universe["date"])

print(f"\n{'='*60}")
print(f"Point-in-Time 유니버스 구축 완료")
print(f"{'='*60}")
print(f"  기간: {pit_universe['date'].min().date()} ~ {pit_universe['date'].max().date()}")
print(f"  총 행: {len(pit_universe):,}")
print(f"  월별 평균 종목 수: {pit_universe.groupby('date')['ticker_yahoo'].count().mean():.0f}")

# 월별 종목 수 추이
monthly_count = pit_universe.groupby("date")["ticker_yahoo"].count()
print(f"\n  종목 수 추이 (5년 간격):")
for yr in [2010, 2013, 2015, 2018, 2020, 2023, 2025]:
    mask = monthly_count.index.year == yr
    if mask.any():
        avg = monthly_count[mask].mean()
        print(f"    {yr}: 평균 {avg:.0f}종목")

# ─── 5. 현재 유니버스 대비 차이 확인 ──────────────────────────────────────────

# 2013-06 기준 PIT vs 현재
pit_2013 = set(pit_universe[pit_universe["date"] == "2013-06-28"]["ticker_yahoo"])
only_current = current_tickers - pit_2013  # 현재는 있지만 2013에는 없었던 종목
only_2013 = pit_2013 - current_tickers      # 2013에는 있었지만 현재는 퇴출된 종목

print(f"\n  2013-06 vs 현재 비교:")
print(f"    현재 S&P500: {len(current_tickers)}종목")
print(f"    2013-06 PIT: {len(pit_2013)}종목")
print(f"    현재만 있음 (2013엔 없었음): {len(only_current)}종목")
print(f"    2013만 있음 (현재 퇴출됨): {len(only_2013)}종목")

if len(only_current) > 0:
    print(f"\n    현재만 있는 종목 예시 (최대 20개):")
    for t in sorted(only_current)[:20]:
        print(f"      {t}")

if len(only_2013) > 0:
    print(f"\n    2013만 있는 종목 예시 (최대 20개):")
    for t in sorted(only_2013)[:20]:
        print(f"      {t}")

# ─── 6. 저장 ──────────────────────────────────────────────────────────────────

pit_universe.to_parquet(INTERIM / "sp500_pit_universe.parquet", index=False)
print(f"\n✅ 저장: sp500_pit_universe.parquet")
print(f"   {len(pit_universe):,} rows")

현재 S&P500: 503종목
변경 이력: 389건 (1976-07-01 ~ 2026-02-09)


PIT Universe: 100%|██████████| 194/194 [00:00<00:00, 66364.49it/s]


Point-in-Time 유니버스 구축 완료
  기간: 2010-01-31 ~ 2026-02-28
  총 행: 98,199
  월별 평균 종목 수: 506

  종목 수 추이 (5년 간격):
    2010: 평균 510종목
    2013: 평균 507종목
    2015: 평균 508종목
    2018: 평균 506종목
    2020: 평균 506종목
    2023: 평균 503종목
    2025: 평균 503종목

  2013-06 vs 현재 비교:
    현재 S&P500: 503종목
    2013-06 PIT: 0종목
    현재만 있음 (2013엔 없었음): 503종목
    2013만 있음 (현재 퇴출됨): 0종목

    현재만 있는 종목 예시 (최대 20개):
      A
      AAPL
      ABBV
      ABNB
      ABT
      ACGL
      ACN
      ADBE
      ADI
      ADM
      ADP
      ADSK
      AEE
      AEP
      AES
      AFL
      AIG
      AIZ
      AJG
      AKAM

✅ 저장: sp500_pit_universe.parquet
   98,199 rows


In [7]:
pit = pd.read_parquet(r"C:\QP2\data\interim\sp500_pit_universe.parquet")
print(pit.columns.tolist())
print(pit.dtypes)
print(pit.head())

['date', 'ticker_yahoo']
date            datetime64[us]
ticker_yahoo               str
dtype: object
        date ticker_yahoo
0 2010-01-31            A
1 2010-01-31           AA
2 2010-01-31         AAPL
3 2010-01-31          ABT
4 2010-01-31          ACE
